# RSNA

## Assignment

---

Sviluppare un modello in grado di facilitare l'identificazione e la classificazione delle condizioni degenerative della colonna vertebrale, utilizzando immagini della colonna lombare acquisite tramite risonanza magnetica.

### Overview

---


Il progetto si propone di classificare cinque principali patologie degenerative della colonna lombare:

1. Stenosi del canale
2. Restringimento del forame sul lato sinistro
3. Restringimento del forame sul lato destro
4. Stenosi subarticolare sul lato sinistro
5. Stenosi subarticolare sul lato destro

Le condizioni descritte possono manifestarsi a diversi livelli della colonna vertebrale, in particolare interessando i seguenti dischi intervertebrali:

1. Tra la prima e la seconda vertebra lombare (L1-L2)
2. Tra la seconda e la terza vertebra lombare (L2-L3)
3. Tra la terza e la quarta vertebra lombare (L3-L4)
4. Tra la quarta e la quinta vertebra lombare (L4-L5)
5. Tra la quinta vertebra lombare e la prima vertebra sacrale (L5-S1)

Inoltre, ogni patologia viene classificata in base al grado di compressione:

1. Normale/Lieve
2. Moderato
3. Grave

Ciascuna patologia è indipendente dalle altre e per ognuna di esse si dovrà calcolare un punteggio per ogni grado di compressione che sia compreso tra 0 e 1. Questo esprime la probabilità che il paziente presenti quella specifica patologia, con il relativo livello di gravità.

## Dataset Description
---

Il dataframe **`train.csv`** contiene le seguenti colonne:

* `study_id` - L'identificativo dello studio. Ciascuno studio può includere più serie di immagini.
* `[condition]_[level]` - Corrisponde a 25 colonne dove vengono combinate:
 
 Tutte le possibili condizioni:
 * `spinal_canal_stenosis` - Stenosi del canale
 * `left_neural_foraminal_narrowing` - Restringimento del forame sul lato sinistro
 * `right_neural_foraminal_narrowing` -  Restringimento del forame sul lato destro
 * `left_subarticular_stenosis` - Stenosi subarticolare sul lato sinistro
 * `right_subarticular_stenosis` - Stenosi subarticolare sul lato destro
 
 <br>
 Con tutti i possibili livelli della colonna vertebrale:
 <br>
 <br>
 
 * `l1_l2` - Disco intervertebrale situato tra le vertebre L1 ed L2
 * `l2_l3` - Disco intervertebrale situato tra le vertebre L2 ed L3
 * `l3_l4` - Disco intervertebrale situato tra le vertebre L3 ed L4
 * `l4_l5` - Disco intervertebrale situato tra le vertebre L4 ed L5
 * `l5_s1` - Disco intervertebrale situato tra le vertebre L5 ed S1

<br>

Questo dataframe mette in relazione, ciascuno studio, per ciascuna delle 25 colonne `[condition]_[level]`, con il relativo grado di compressione:

* `Normal/Mild`
* `Moderate`
* `Severe`

<br>

Il dataframe presenta valori mancanti in alcune delle colonne `[condition]_[level]`.

---

I dataframe **`train_series_descriptions.csv`** e **`test_series_descriptions.csv`** contengono le seguenti colonne:

* `study_id` - L'identificativo dello studio
* `series_id` - L'identificativo della serie
* `series_description` - La tipologia della scansione (Axial T2, Sagittal T1, Sagittal T2/STIR)

I dataframe non presentano valori mancanti.

---

Il dataframe **`train_label_coordinates.csv`** contiene le seguenti colonne:

* `study_id` - L'identificativo dello studio
* `series_id` - L'identificativo della serie
* `instance_number` - L'ordine dell'immagine all'interno dello stack 3D.
* `condition` - La condizione
* `level` - Il livello all'interno della colonna vertebrale
* `x`, `y` - Le coordinate x ed y del centro dell'area che definisce l'etichetta

Il dataframe non presenta valori mancanti.

---

Le cartelle **`train_images`** e **`test_images`** sono organizzate come segue:

&emsp;&emsp; `[train/test]_images/[study_id]/[series_id]/[instance_number].dcm`

Ciascun file `.dcm` rappresenta un'immagine acquisita tramite risonanza magnetica di tipo _Digital Imaging and Communications in Medicine_.

---

I dataframe **`sample_submission.csv`** rappresenta un esempio dell'output desiderato:

* `row_id` - Un'identificativo ottenuto dalla combinazione di `study_id`, `condition` e `level`, ad esempio `12345_spinal_canal_stenosis_l3_l4`
* `normal_mild`, `moderate`, `severe` - Le 3 colonne relative alle predizioni, che rappresentano la probabilità che il paziente presenti quel particolare livello di gravità

<br>

In [230]:
import os

from math import cos, radians, sin
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import visualkeras

from PIL import Image

import pydicom
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix

import tensorflow as tf

from tensorflow.keras import backend as K
from tensorflow.keras import Input, Model, layers, models
from tensorflow.keras.optimizers import Adam

from IPython.display import clear_output
import warnings

In [231]:
df_diagnosis = pd.read_csv('Dati/train.csv')
print("Total rows:", len(df_diagnosis))
df_diagnosis.head()

Total rows: 1975


,study_id,spinal_canal_stenosis_l1_l2,spinal_canal_stenosis_l2_l3,spinal_canal_stenosis_l3_l4,spinal_canal_stenosis_l4_l5,spinal_canal_stenosis_l5_s1,left_neural_foraminal_narrowing_l1_l2,left_neural_foraminal_narrowing_l2_l3,left_neural_foraminal_narrowing_l3_l4,left_neural_foraminal_narrowing_l4_l5,...,left_subarticular_stenosis_l1_l2,left_subarticular_stenosis_l2_l3,left_subarticular_stenosis_l3_l4,left_subarticular_stenosis_l4_l5,left_subarticular_stenosis_l5_s1,right_subarticular_stenosis_l1_l2,right_subarticular_stenosis_l2_l3,right_subarticular_stenosis_l3_l4,right_subarticular_stenosis_l4_l5,right_subarticular_stenosis_l5_s1
0,4003253,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Moderate,...,Normal/Mild,Normal/Mild,Normal/Mild,Moderate,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild
1,4646740,Normal/Mild,Normal/Mild,Moderate,Severe,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Moderate,...,Normal/Mild,Normal/Mild,Normal/Mild,Severe,Normal/Mild,Normal/Mild,Moderate,Moderate,Moderate,Normal/Mild
2,7143189,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,...,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild
3,8785691,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Moderate,...,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild
4,10728036,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,...,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Normal/Mild,Moderate,Normal/Mild


In [232]:
df_series = pd.read_csv('Dati/train_series_descriptions.csv')
print("Total rows:", len(df_series))
df_series.head()

Total rows: 6294


,study_id,series_id,series_description
0,4003253,702807833,Sagittal T2/STIR
1,4003253,1054713880,Sagittal T1
2,4003253,2448190387,Axial T2
3,4646740,3201256954,Axial T2
4,4646740,3486248476,Sagittal T1


In [233]:
df_instances = pd.read_csv('Dati/train_label_coordinates.csv')
print("Total rows:", len(df_instances))
df_instances.head()

Total rows: 48692


,study_id,series_id,instance_number,condition,level,x,y
0,4003253,702807833,8,Spinal Canal Stenosis,L1/L2,322.831858,227.964602
1,4003253,702807833,8,Spinal Canal Stenosis,L2/L3,320.571429,295.714286
2,4003253,702807833,8,Spinal Canal Stenosis,L3/L4,323.030303,371.818182
3,4003253,702807833,8,Spinal Canal Stenosis,L4/L5,335.292035,427.327434
4,4003253,702807833,8,Spinal Canal Stenosis,L5/S1,353.415929,483.964602


In [ ]:
def get_scan(file_path):
    dicom_data = pydicom.dcmread(file_path)
    return dicom_data.pixel_array

In [ ]:
for series_description in ["Axial T2", "Sagittal T1", "Sagittal T2/STIR"]:
    series_row = df_series[df_series["series_description"] == series_description].iloc[0]
    
    study_id = series_row["study_id"]
    series_id = series_row["series_id"]

    instance_row = df_instances[(df_instances["study_id"] == study_id) & (df_instances["series_id"] == series_id)].iloc[0]
    instance_number = instance_row["instance_number"]
    x = instance_row["x"]
    y = instance_row["y"]

    file_path = f"Dati/train_images/{study_id}/{series_id}/{instance_number}.dcm"

    scan = get_scan(file_path)

    plt.imshow(scan, cmap='gray')
    plt.axis('off')
    plt.show()

# Data Analysis

Ogni paziente è associato a uno studio, mentre ad ogni studio sono associate una o più serie di immagini, ciascuna appartenente a una delle seguenti tipologie: "Axial T2", "Sagittal T1" o "Sagittal T2/STIR". Ogni serie include una sequenza di immagini chiamate "istanze".

Per ogni studio, vengono diagnosticate le cinque possibili patologie della colonna vertebrale, ciascuna delle quali può manifestarsi in uno dei cinque livelli vertebrali, per un totale di 25 combinazioni possibili. Le diagnosi possono essere classificate come "Normal/Mild", "Moderate" o "Severe".

Ad ogni immagine (istanza) è associata una o più delle 25 patologie tramite:

> La colonna "condition", che può riportare uno dei seguenti valori:
> 1. Spinal Canal Stenosis
> 2. Right Neural Foraminal Narrowing
> 3. Left Neural Foraminal Narrowing
> 4. Right Subarticular Stenosis
> 5. Left Subarticular Stenosis

> E la colonna "level", che identifica il livello vertebrale coinvolto:
> 1. L1/L2
> 2. L2/L3
> 3. L3/L4
> 4. L4/L5
> 5. L5/S1

Per ciascuna patologia associata vengono inoltre fornite le coordinate x ed y, che indicano con precisione il punto in cui si può osservare il manifestarsi (o meno) della condizione.

Si intende sviluppare un modello basato su una rete neurale convoluzionale che metta in relazione ad ogni immagine ed ai valori delle coordinate delle patologie in evidenza nella stessa, un vettore binario multilabel che esprima le diagnosi associate all'immagine: 0 per "Normal/Mild" e 1 per "Moderate/Severe".

# Feature Engineering

I dati verranno riorganizzati nel dataframe **`df_total`** che includerà le seguenti colonne:

* `study_id` - L'identificativo dello studio.
* `series_id` - L'identificativo della serie.
* `instance_number` - L'identificativo dell'istanza (immagine).
* `condition_level` - Il vettore delle patologie associate all'istanza `instance_number` ottenuto combinando i valori delle colonne `condition` e `level` del dataframe `df_instances` e formattandoli come le colonne del dataframe `df_diagnosis`.
* `coor` - La tupla contenente le coordinate x ed y associate all'istanza `instance_number` nel dataframe `df_instances`.
* `diagnosis` - Il vettore delle diagnosi relative allo studio `study_id` nel dataframe `df_diagnosis` in corrispondenza delle patologie elencate nel vettore `condition_level`. Tuttavia, le diagnosi "Moderate" e "Severe" verranno accorpate in un'unica categoria, "Moderate/Severe", per semplificare il processo di classificazione.

Gli studi con diagnosi incomplete non saranno inclusi nel dataframe.

In [ ]:
df_total = pd.DataFrame({
    "study_id": [],
    "series_id": [],
    "instance_number": [],
    "condition_level": [],
    "coor": [],
    "diagnosis": []
})

df_total["study_id"] = df_total["study_id"].astype(int)
df_total["series_id"] = df_total["series_id"].astype(int)
df_total["instance_number"] = df_total["instance_number"].astype(int)

remove_row = []

progress = 0

study_id_list = list(set(df_instances["study_id"]))
for study_id in study_id_list:
    
    clear_output(wait=True)
    print(f"{round(progress / len(study_id_list) * 100, 2)}%")
    
    series_id_list = list(set(df_instances["series_id"][df_instances["study_id"] == study_id]))
    for series_id in series_id_list:
        instance_list = list(set(df_instances["instance_number"][(df_instances["study_id"] == study_id) & (df_instances["series_id"] == series_id)]))
        for instance_number in instance_list:
            df = df_instances[(df_instances["study_id"] == study_id) & (df_instances["series_id"] == series_id) & (df_instances["instance_number"] == instance_number)]

            condition_level = []
            coor = []
            diagnosis = []

            for row_index, row in df.iterrows():
                condition_level.append(row["condition"].lower().replace(" ", "_") + "_" + row["level"].lower().replace("/", "_"))
                coor.append((row["x"], row["y"]))
                
                diag = df_diagnosis[condition_level[-1]][df_diagnosis["study_id"] == row["study_id"]].iloc[0]
                
                if diag == "Normal/Mild":
                    diagnosis.append("Normal/Mild")
                elif diag == "Moderate" or diag == "Severe":
                    diagnosis.append("Moderate/Severe")
                else:
                    diagnosis.append("Missing")
                    remove_row.append(len(df_total))
            
            new_row = pd.DataFrame({
                "study_id": [int(study_id)],
                "series_id": [int(series_id)],
                "instance_number": [int(instance_number)],
                "condition_level": [condition_level],
                "coor": [coor],
                "diagnosis": [diagnosis]
            })

            df_total = pd.concat([df_total, new_row], ignore_index=True)
    
    progress += 1

print(f"{len(remove_row)} rows removed")
df_total = df_total.drop(remove_row)

df_total = df_total.sort_values(by=list(df_total.columns[0:2]))

print("Total rows:", len(df_total))

df_total.head()

Poiché i tre tipi di imaging — "Axial T2", "Sagittal T1" e "Sagittal T2/STIR" — hanno strutture e composizioni diverse, il dataframe `df_total` verrà suddiviso in tre dataframe distinti:
1. **`df_axial_t2`**: conterrà le serie di tipo "Axial T2".
2. **`df_sagittal_t1`**: conterrà le serie di tipo "Sagittal T1".
3. **`df_sagittal_t2`**: conterrà le serie di tipo "Sagittal T2/STIR".

In [ ]:
df_axial_t2 = df_total[df_total["series_id"].isin(list(df_series["series_id"][df_series["series_description"] == "Axial T2"]))].copy()

print("Total rows:", len(df_axial_t2))
df_axial_t2.head()

In [ ]:
df_sagittal_t1 = df_total[df_total["series_id"].isin(list(df_series["series_id"][df_series["series_description"] == "Sagittal T1"]))].copy()

print("Total rows:", len(df_sagittal_t1))
df_sagittal_t1.head()

In [ ]:
df_sagittal_t2 = df_total[df_total["series_id"].isin(list(df_series["series_id"][df_series["series_description"] == "Sagittal T2/STIR"]))].copy()

print("Total rows:", len(df_sagittal_t2))
df_sagittal_t2.head()

In [ ]:
def plot_diagnosis_dist(df, title=""):
    
    labels = list(df_diagnosis.columns[1:26])
    
    dist = {
        "Normal/Mild": [0] * len(labels),
        "Moderate/Severe": [0] * len(labels)
    }
    
    for label_index in range(len(labels)):
        label = labels[label_index]
        label_diagnosis = [item for sublist in list(df["diagnosis"][df["condition_level"].apply(lambda x: label in x)]) for item in sublist]
        for diagnosis in label_diagnosis:
            dist[diagnosis][label_index] += 1
    
    fig = plt.figure(figsize = (12, 5))
    x = np.arange(len(labels))

    plt.bar(x, dist["Normal/Mild"], label = "Normal/Mild", color='C0', width = 0.5)
    plt.bar(x, dist["Moderate/Severe"], label = "Moderate/Severe", bottom = dist["Normal/Mild"], color='C1', width = 0.5)
    
    max_value = max(dist["Normal/Mild"] + dist["Moderate/Severe"])
    label_colors = []
    
    for i in range(len(labels)):
        normal_mild_value = dist["Normal/Mild"][i]
        moderate_severe_value = dist["Moderate/Severe"][i]
        total_value = normal_mild_value + moderate_severe_value
        
        if total_value == 0:
            label_colors.append("red")
        else:
            label_colors.append("black")
        
        if total_value > max_value * 0.10:
            plt.text(i, total_value / 2, f"{normal_mild_value} / {moderate_severe_value}", ha='center', va='center', color='white', fontsize=8, fontweight='bold', rotation=90)
        else:
            plt.text(i, total_value + max_value * 0.05, f"{normal_mild_value} / {moderate_severe_value}", ha='center', va='bottom', color='black', fontsize=8, rotation=90)
    
    plt.title(title)
    plt.ylabel("Instances Count")
    
    plt.xticks(x, labels, rotation=90)
    ax = plt.gca()  # Ottieni l'asse corrente
    for tick_label, color in zip(ax.get_xticklabels(), label_colors):
        tick_label.set_color(color)
    
    plt.xlim(-0.5, len(labels) - 0.5)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

    plt.show()

In [ ]:
plot_diagnosis_dist(df_axial_t2, "Axial T2 Diagnosis")
plot_diagnosis_dist(df_sagittal_t1, "Sagittal T1 Diagnosis")
plot_diagnosis_dist(df_sagittal_t2, "Sagittal T2/STIR Diagnosis")

Osservando la distribuzione delle diagnosi "Moderate/Severe" rispetto a quelle "Normal/Mild", emerge un evidente sbilanciamento in alcune patologie.

Inoltre, dall'analisi della distribuzione delle diagnosi in base ai diversi tipi di serie, emergono alcune osservazioni interessanti:
* Le serie di tipo "Axial T2" sono esclusivamente associate alle diagnosi di "Subarticular Stenosis".
* Le serie di tipo "Sagittal T1" sono prevalentemente collegate alla diagnosi di "Neural Foraminal Narrowing", con solo 5 casi relativi a "Spinal Canal Stenosis".
* Le serie di tipo "Sagittal T2/STIR" risultano associate unicamente alla diagnosi di "Spinal Canal Stenosis".

Questo quadro mostra una chiara correlazione tra il tipo di imaging e le patologie diagnosticate.

# Data Augmentation

Al fine di bilanciare la classe "Moderate/Severe" con quella "Normal/Mild" si prevede di utilizzare il Data Augmentation. Tuttavia, poiché questo processo è stocastico e non consente di monitorare con precisione le trasformazioni delle coordinate in relazione all'immagine, è stato sviluppato un sistema personalizzato per ottenere un maggiore controllo e tenere traccia delle modifiche.

Le trasformazioni sono le seguenti:
* Flip
* Rotazione
* Traslazione
* Zoom

In [ ]:
random.seed(380905)

def get_flip():
    return "horizontal"

def get_rotation():
    return random.randrange(-10, 10 + 1)

def get_translation():
    return (random.randrange(-100, 100 + 1), random.randrange(-100, 100 + 1))

def get_zoom():
    return random.randrange(7, 12 + 1) / 10 # 0.91 - 1.56

In [ ]:
def apply_flip_to_scan(scan, flip):
    if flip == "horizontal":
        return np.fliplr(scan)
    elif flip == "vertical":
        return np.flipud(scan)
    else:
        return scan


def apply_rotation_to_scan(scan, rotation):
    img = Image.fromarray(scan)
    img = img.rotate(rotation, expand=False)
    return np.array(img)


def apply_translation_to_scan(scan, translation):
    height, width = scan.shape[:2]
    tx, ty = translation
    
    trans_x = (0, width)
    trans_y = (0, height)
    scan_x = (0, width)
    scan_y = (0, height)
    
    if ty > 0:
        trans_y = (ty, height)
        scan_y = (0, height - ty)
    elif ty < 0:
        trans_y = (0, height + ty)
        scan_y = (-ty, height)
    
    if tx > 0:
        trans_x = (tx, width)
        scan_x = (0, width - tx)
    elif tx < 0:
        trans_x = (0, width + tx)
        scan_x = (-tx, width)
    
    translated_scan = np.zeros_like(scan)
    translated_scan[trans_y[0]:trans_y[1], trans_x[0]:trans_x[1]] = scan[scan_y[0]:scan_y[1], scan_x[0]:scan_x[1]]
    return translated_scan


def apply_zoom_to_scan(scan, zoom):
    if zoom == 1:
        return scan

    height, width = scan.shape[:2]
    
    scaled_height, scaled_width = int(height * zoom), int(width * zoom)
    scaled_scan = np.array(Image.fromarray(scan).resize((scaled_width, scaled_height)))
    
    start_y = (height - scaled_height) // 2
    start_x = (width - scaled_width) // 2
    
    if zoom > 1: # Zoom In
        start_y = max(0, -start_y)
        start_x = max(0, -start_x)
        zoomed_scan = scaled_scan[start_y:start_y + height, start_x:start_x + width]
        return zoomed_scan
    
    elif zoom < 1: # Zoom Out
        zoomed_scan = np.zeros_like(scan)
        
        start_y = max(0, start_y)
        start_x = max(0, start_x)
        zoomed_scan[start_y:start_y + scaled_height, start_x:start_x + scaled_width] = scaled_scan
        return zoomed_scan

In [ ]:
def apply_transform_to_scan(scan, transform):
    
    if transform != "none":
        transform_data = transform.split(";")

        flip = transform_data[0]
        rotation = int(transform_data[1])
        translation = eval(transform_data[2])
        zoom = float(transform_data[3])
        
        scan = apply_flip_to_scan(scan, flip)
        scan = apply_rotation_to_scan(scan, rotation)
        scan = apply_translation_to_scan(scan, translation)
        scan = apply_zoom_to_scan(scan, zoom)
    
    return scan

In [ ]:
def apply_flip_to_coor(x, y, scan, flip):
    height, width = scan.shape[:2]
    
    if flip == "horizontal":
        x = width - 1 - x
    elif flip == "vertical":
        y = height - 1 - y

    return x, y


def apply_rotation_to_coor(x, y, scan, rotation):
    height, width = scan.shape[:2]
    
    center_x, center_y = width // 2, height // 2
    
    translated_x = x - center_x
    translated_y = y - center_y
    
    angle_rad = np.radians(rotation * -1)
    
    rotated_x = translated_x * np.cos(angle_rad) - translated_y * np.sin(angle_rad)
    rotated_y = translated_x * np.sin(angle_rad) + translated_y * np.cos(angle_rad)
    
    expanded_center_x, expanded_center_y = center_x, center_y
    
    new_x = int(rotated_x + expanded_center_x)
    new_y = int(rotated_y + expanded_center_y)
    
    return new_x, new_y


def apply_zoom_to_coor(x, y, scan, zoom):
    height, width = scan.shape[:2]
    center_x, center_y = width // 2, height // 2
    
    new_x = center_x + (x - center_x) * zoom
    new_y = center_y + (y - center_y) * zoom
    
    return int(new_x), int(new_y)


def apply_translation_to_coor(x, y, scan, translation):
    tx, ty = translation
    new_x = x + tx
    new_y = y + ty
    
    height, width = scan.shape[:2]
    new_x = np.clip(new_x, 0, width - 1)
    new_y = np.clip(new_y, 0, height - 1)
    
    return new_x, new_y

Oltre alle trasformazioni previste dal Data Augmentation, è stato implementato un sistema di crop che ritaglia l'immagine di un determinato valore scalare intorno al suo centro.


def crop_scan(scan, zoom = 1.3):
    if zoom <= 1:
        return scan
    
    center_x = scan.shape[1] / 2
    center_y = scan.shape[0] / 2
    
    rect_width = scan.shape[1] / zoom
    rect_height = scan.shape[0] / zoom
    
    x0 = center_x - rect_width / 2
    x1 = center_x + rect_width / 2
    y0 = center_y - rect_height / 2
    y1 = center_y + rect_height / 2
    
    if x0 < 0:
        x0 = 0
        x1 = rect_width
    elif x1 > scan.shape[1]:
        x0 = scan.shape[1] - rect_width
        x1 = scan.shape[1]
    
    if y0 < 0:
        y0 = 0
        y1 = rect_height
    elif y1 > scan.shape[1]:
        y0 = scan.shape[0] - rect_height
        y1 = scan.shape[0]
    
    x0 = int(x0)
    x1 = int(x1)
    y0 = int(y0)
    y1 = int(y1)
    
    new_scan = scan[y0:y1, x0:x1]
        
    return new_scan
    

In [ ]:
def crop_scan(scan, zoom = 1.3):
    if zoom <= 1:
        return scan
    
    center_x = scan.shape[1] / 2
    center_y = scan.shape[0] / 2
    
    rect_width = scan.shape[1] / zoom
    rect_height = scan.shape[0] / zoom
    
    x0 = center_x - rect_width / 2
    x1 = center_x + rect_width / 2
    y0 = center_y - rect_height / 2
    y1 = center_y + rect_height / 2
    
    if x0 < 0:
        x0 = 0
        x1 = rect_width
    elif x1 > scan.shape[1]:
        x0 = scan.shape[1] - rect_width
        x1 = scan.shape[1]
    
    if y0 < 0:
        y0 = 0
        y1 = rect_height
    elif y1 > scan.shape[1]:
        y0 = scan.shape[0] - rect_height
        y1 = scan.shape[0]
    
    x0 = int(x0)
    x1 = int(x1)
    y0 = int(y0)
    y1 = int(y1)
    
    new_scan = scan[y0:y1, x0:x1]
        
    return new_scan

In [ ]:
def apply_crop_to_coor(x, y, source_scan, zoom = 1.3):
    if zoom <= 1:
        return x, y
    
    center_x = source_scan.shape[1] / 2
    center_y = source_scan.shape[0] / 2
    
    rect_width = source_scan.shape[1] / zoom
    rect_height = source_scan.shape[0] / zoom
    
    x_offset = center_x - rect_width / 2
    y_offset = center_y - rect_height / 2
    
    return (x - x_offset), (y - y_offset)

In [ ]:
instance_row = df_instances.iloc[0]

study_id = instance_row["study_id"]
series_id = instance_row["series_id"]
instance_number = instance_row["instance_number"]
x = instance_row["x"]
y = instance_row["y"]

file_path = f"Dati/train_images/{study_id}/{series_id}/{instance_number}.dcm"


scan = get_scan(file_path)

print("Source Image")

plt.imshow(scan, cmap='gray')
plt.axis('off')
plt.scatter([x], [y], color='red', s=10)
plt.show()
print("Size:", scan.shape[:2])
print("----------------------------------------------------------------")

scan = apply_flip_to_scan(scan, "horizontal")
x, y = apply_flip_to_coor(x, y, scan, "horizontal")

print("Flip")

plt.imshow(scan, cmap='gray')
plt.axis('off')
plt.scatter([x], [y], color='red', s=10)
plt.show()
print("Size:", scan.shape[:2])
print("----------------------------------------------------------------")

scan = apply_rotation_to_scan(scan, 10)
x, y = apply_rotation_to_coor(x, y, scan, 10)

print("Rotation")

plt.imshow(scan, cmap='gray')
plt.axis('off')
plt.scatter([x], [y], color='red', s=10)
plt.show()
print("Size:", scan.shape[:2])
print("----------------------------------------------------------------")

scan = apply_translation_to_scan(scan, (100, 100))
x, y = apply_translation_to_coor(x, y, scan, (100, 100))

print("Translation")

plt.imshow(scan, cmap='gray')
plt.axis('off')
plt.scatter([x], [y], color='red', s=10)
plt.show()
print("Size:", scan.shape[:2])
print("----------------------------------------------------------------")

scan = apply_zoom_to_scan(scan, 1.2)
x, y = apply_zoom_to_coor(x, y, scan, 1.2)

print("Zoom")

plt.imshow(scan, cmap='gray')
plt.axis('off')
plt.scatter([x], [y], color='red', s=10)
plt.show()
print("Size:", scan.shape[:2])
print("----------------------------------------------------------------")

x, y = apply_crop_to_coor(x, y, scan)
scan = crop_scan(scan)

print("Crop")

plt.imshow(scan, cmap='gray')
plt.axis('off')
plt.scatter([x], [y], color='red', s=10)
plt.show()
print("Size:", scan.shape[:2])
print("----------------------------------------------------------------")

La procedura di Data Augmentation ordina le righe dando priorità a quelle della classe minoritaria e seleziona le prime N righe, dove N = Righe Totali / `damping_factor`.

Inoltre, aggiunge al dataframe le colonne `transform_id`, per distinguere i duplicati di una stessa istanza, e `transform`, per memorizzare le trasformazioni che verranno applicate all'immagine, che saranno memorizzate come una stringa formattata come segue:

`transform = str(flip) + ";" + str(rotation) + ";" + str(translation) + ";" + str(zoom)`

Infine, le trasformazioni vengono applicate anche alle coordinate.

In [ ]:
def data_augment(df, labels, damping_factor=2):
    
    df_augment = df.copy()
    
    if not "transform_id" in df_augment.columns:
        df_augment["transform_id"] = 0
        cols = list(df_augment.columns)
        cols.insert(3, cols.pop(cols.index("transform_id")))
        df_augment = df_augment[cols]
    
    if not "transform" in df_augment.columns:
        df_augment["transform"] = "none"
        cols = list(df_augment.columns)
        cols.insert(4, cols.pop(cols.index("transform")))
        df_augment = df_augment[cols]
    
    df_augment["study_id"] = df_augment["study_id"].astype(int)
    df_augment["series_id"] = df_augment["series_id"].astype(int)
    df_augment["instance_number"] = df_augment["instance_number"].astype(int)
    df_augment["transform_id"] = df_augment["transform_id"].astype(int)
    
    for label in labels:
        
        df_label = df_augment[df_augment["condition_level"].apply(lambda x: isinstance(x, list) and label in x)].copy()
        
        df_normal_mild = df_label[df_label["diagnosis"].apply(lambda x: "Normal/Mild" in x)].copy()
        df_moderate_severe = df_label[df_label["diagnosis"].apply(lambda x: "Moderate/Severe" in x)].copy()
        
        def diagnosis_priority(row, diagnosis, label):
            try:
                label_index = row['condition_level'].index(label)
                return row['diagnosis'][label_index] == diagnosis
            except ValueError:
                return False
        
        df_normal_mild["priority"] = df_normal_mild.apply(diagnosis_priority, axis=1, args=("Normal/Mild", label))
        df_normal_mild["diagnosis_count"] = df_normal_mild["diagnosis"].apply(lambda x: x.count("Normal/Mild"))
        df_normal_mild = df_normal_mild.sort_values(by=["priority", "diagnosis_count"], ascending=[False, False])
        
        df_moderate_severe["priority"] = df_moderate_severe.apply(diagnosis_priority, axis=1, args=("Moderate/Severe", label))
        df_moderate_severe["diagnosis_count"] = df_moderate_severe["diagnosis"].apply(lambda x: x.count('Moderate/Severe'))
        df_moderate_severe = df_moderate_severe.sort_values(by=["priority", "diagnosis_count"], ascending=[False, False])
        
        df_majority = df_normal_mild.copy()
        df_minority = df_moderate_severe.copy()
        
        if len(df_normal_mild) < len(df_moderate_severe):
            df_majority = df_moderate_severe.copy()
            df_minority = df_normal_mild.copy()
        
        augment_size = int(len(df_majority) / len(df_minority))
        
        progress = 0
        for row_index, row in df_minority.iterrows():
            
            if progress > len(df_minority) / damping_factor:
                break
            
            for i in range(augment_size):
                
                study_id = row["study_id"]
                series_id = row["series_id"]
                instance_number = row["instance_number"]
                
                transform_id = max(list(df_augment["transform_id"][(df_augment["study_id"] == study_id) & (df_augment["series_id"] == series_id) & (df_augment["instance_number"] == instance_number)])) + 1
                
                flip = get_flip()
                rotation = get_rotation()
                zoom = get_zoom()
                translation = get_translation()
                transform = str(flip) + ";" + str(rotation) + ";" + str(translation) + ";" + str(zoom)
                
                scan = get_scan(f"Dati/train_images/{study_id}/{series_id}/{instance_number}.dcm")
                
                coor = []
                for i in range(len(row["coor"])):
                    x = row["coor"][i][0]
                    y = row["coor"][i][1]
                    
                    x, y = apply_flip_to_coor(x, y, scan, flip)
                    x, y = apply_rotation_to_coor(x, y, scan, rotation)
                    x, y = apply_translation_to_coor(x, y, scan, translation)
                    x, y = apply_zoom_to_coor(x, y, scan, zoom)
                    x, y = apply_crop_to_coor(x, y, scan)
                    
                    coor.append((x, y))
                
                new_row = pd.DataFrame({
                    "study_id": [int(row["study_id"])],
                    "series_id": [int(row["series_id"])],
                    "instance_number": [int(row["instance_number"])],
                    "transform_id": [int(transform_id)],
                    "transform": [transform],
                    "condition_level": [row["condition_level"]],
                    "coor": [coor],
                    "diagnosis": [row["diagnosis"]]
                })
                
                df_augment = pd.concat([df_augment, new_row], ignore_index=True)
            progress += 1
    
    clear_output(wait=True)
    return df_augment

È stato adottato il dataframe `df_sagittal_t2` perché, a differenza degli altri, presenta solo 5 patologie, semplificando così la classificazione e rendendola più agevole per la rete neurale.

In [ ]:
df_sagittal_t2_augment = data_augment(df_sagittal_t2, list(df_diagnosis.columns[1:6]), 20)
df_sagittal_t2_augment = data_augment(df_sagittal_t2_augment, list(df_diagnosis.columns[1:6]), 20)
df_sagittal_t2_augment = data_augment(df_sagittal_t2_augment, list(df_diagnosis.columns[1:6]), 20)

In [ ]:
print("Total rows:", len(df_sagittal_t2_augment), f"({(len(df_sagittal_t2_augment) - len(df_sagittal_t2))} rows added)")
df_sagittal_t2_augment.head()

Dopo la procedura di Data Augmentation, le coordinate vengono normalizzate.

In [ ]:
def data_normalization(df):
    
    df_normal = pd.DataFrame({
        "study_id": [], 
        "series_id": [], 
        "instance_number": [], 
        "transform_id": [], 
        "transform": [],
        "condition_level": [], 
        "coor": [], 
        "diagnosis": []
    })

    df_normal["study_id"] = df_normal["study_id"].astype(int)
    df_normal["series_id"] = df_normal["series_id"].astype(int)
    df_normal["instance_number"] = df_normal["instance_number"].astype(int)
    df_normal["transform_id"] = df_normal["transform_id"].astype(int)
    
    progress = 0
    for row_index, row in df.iterrows():
        
        clear_output(wait=True)
        print(f"{round(progress / len(df) * 100, 2)}%")
        
        study_id = row["study_id"]
        series_id = row["series_id"]
        instance_number = row["instance_number"]
        
        scan = get_scan(f"Dati/train_images/{study_id}/{series_id}/{instance_number}.dcm")
        scan = crop_scan(scan)
        height, width = scan.shape[:2]
        
        coor = []
        for i in range(len(row["coor"])):
            coor.append((row["coor"][i][0] / width, row["coor"][i][1] / height))
        
        new_row = pd.DataFrame({
            "study_id": [int(study_id)],
            "series_id": [int(series_id)],
            "instance_number": [int(instance_number)],
            "transform_id": [int(row["transform_id"])],
            "transform": [row["transform"]],
            "condition_level": [row["condition_level"]],
            "coor": [coor],
            "diagnosis": [row["diagnosis"]]
        })
        
        df_normal = pd.concat([df_normal, new_row], ignore_index=True)
        
        progress += 1
    
    return df_normal

In [ ]:
df_sagittal_t2_normal = data_normalization(df_sagittal_t2_augment)

In [ ]:
print("Total rows:", len(df_sagittal_t2_normal))
df_sagittal_t2_normal.head()

In [ ]:
df_sagittal_t2_normal[df_sagittal_t2_normal["transform_id"] > 0].head()

In [ ]:
plot_diagnosis_dist(df_sagittal_t2, "Sagittal T2/STIR Diagnosis")
plot_diagnosis_dist(df_sagittal_t2_normal, "Augmented Sagittal T2/STIR Diagnosis")

Osservando le differenze tra i due grafici, si può notare che il bilanciamento è stato efficace, ora le classi "Normal/Mild" e "Moderate/Severe" sono bilanciate.

# Image Size Analysis

Ora procediamo a determinare la distribuzione delle dimensioni delle immagini, analizzando le loro larghezze e altezze per capire la variabilità e adattare eventualmente il preprocessing.

In [ ]:
def get_image_sizings(df):
    
    image_sizings = {}
    
    progress = 0
    for row_index, row in df.iterrows():
        
        clear_output(wait=True)
        print(f"{round(progress / len(df) * 100, 2)}%")
        
        study_id = row["study_id"]
        series_id = row["series_id"]
        instance_number = row["instance_number"]
        
        scan = get_scan(f"Dati/train_images/{study_id}/{series_id}/{instance_number}.dcm")
        height, width = scan.shape[:2]
        key = f"{width}x{height}"
        
        if key in list(image_sizings.keys()):
            image_sizings[key] += 1
        else:
            image_sizings[key] = 1
        
        progress += 1
    
    image_sizings = dict(sorted(image_sizings.items(), key=lambda item: item[1]))
    
    return image_sizings

In [ ]:
image_sizings = get_image_sizings(df_sagittal_t2_normal)

for sizing in list(image_sizings.keys()):
    print(f"[{sizing}]: {image_sizings[sizing]}")

Il formato che rappresenta la maggior parte dei dati è `384x384`, con un totale di 1351 record.

In [ ]:
for row_index, row in df_sagittal_t2_normal.iterrows():
    study_id = row["study_id"]
    series_id = row["series_id"]
    instance_number = row["instance_number"]

    scan = get_scan(f"Dati/train_images/{study_id}/{series_id}/{instance_number}.dcm")
    height, width = scan.shape[:2]
    
    if width == 384 and height == 384:
        print("Source Image")
        plt.imshow(scan, cmap='gray')
        plt.axis('off')
        plt.show()
        print("Size:", scan.shape[:2])
        print("----------------------------------------------------------------")

        print("Crop")
        scan = crop_scan(scan)
        plt.imshow(scan, cmap='gray')
        plt.axis('off')
        plt.show()
        print("Size:", scan.shape[:2])
        print("----------------------------------------------------------------")

        for size in [256, 128, 64]:
            print(f"Resize {size}x{size}")
            scan_img = Image.fromarray(scan)
            scan_img = scan_img.resize((size, size))
            res_scan = np.array(scan_img)
            plt.imshow(res_scan, cmap='gray')
            plt.axis('off')
            plt.show()
            print("Size:", res_scan.shape[:2])
            print("----------------------------------------------------------------")
        
        break

La risoluzione `128x128` sembra essere una scelta adeguata come formato standard per l'input del modello. Ridurre le immagini a questa dimensione consente di ottimizzare il processo di allenamento, riducendo il carico computazionale pur mantenendo abbastanza dettagli per una corretta classificazione. Inoltre, tale formato è comunemente utilizzato per modelli di visione artificiale e potrebbe migliorare l'efficienza del modello.

# Data Train / Test Split

Prima di essere utilizzati nel modello, i dati vengono preprocessati e suddivisi in Train e Test.

Gli input sono:
* "file_path": Il percorso del file dell'immagine.
* "transform": Le trasformazioni che dovranno essere applicate all'immagine.
* "coor": La lista di tuple contenenti le coordinate associate all'immagine, ordinate a seconda dell'ordine delle patologie. I valori mancanti sono sostituiti con (0, 0).

L'output è un vettore che contiene i valori delle diagnosi espressi in formato binario (0 per Normal/Mild e 1 per Moderate/Severe), anch'essi ordinati posizionalmente secondo l'ordine delle patologie. Anche in questo caso, i valori mancanti sono sostituiti con 0.

In [ ]:
def data_process(df, labels):
    
    x_data = {
        "file_path": [],
        "transform": [],
        "coor": []
    }

    y_data = {
        "diagnosis": []
    }
    
    for row_index, row in df.iterrows():
        study_id = row["study_id"]
        series_id = row["series_id"]
        instance_number = row["instance_number"]
        transform_id = row["transform_id"]
        transform = row["transform"]
        
        file_path = f"Dati/train_images/{study_id}/{series_id}/{instance_number}.dcm"
        
        diagnosis = [0] * len(labels)
        coor = [[0, 0]] * len(labels)
        
        for i in range(len(row["condition_level"])):
            index = labels.index(row["condition_level"][i])
            
            coor[index] = [row["coor"][i][0], row["coor"][i][1]]
            
            if row["diagnosis"][i] == "Moderate/Severe":
                diagnosis[index] = 1
        
        x_data["file_path"].append(file_path)
        x_data["transform"].append(transform)
        x_data["coor"].append(coor)
        
        y_data["diagnosis"].append(diagnosis)
    
    # Controllo delle Dimensioni
    prev_key = ""
    for key in list(x_data.keys()):
        size_control = False
        if prev_key == "":
            size_control = len(y_data["diagnosis"]) == len(x_data[key])
        else:
            size_control = len(x_data[prev_key]) == len(x_data[key])
        if not size_control:
            print("ERROR")
            raise KeyboardInterrupt
        prev_key = key
    
    
    x = pd.DataFrame(x_data)
    y = pd.DataFrame(y_data)

    return x, y

Il Train/Test split è stato applicato con una proporzione del 70/30, in modo da garantire un buon equilibrio tra allenamento e validazione del modello.

In [ ]:
labels = list(df_diagnosis.columns[1:6])
x, y = data_process(df_sagittal_t2_normal, labels)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [ ]:
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("")

print("x_train head:\n", x_train.head())
print("")
print("x_test head:\n", x_test.head())
print("")

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("")

print("y_train head:\n", y_train.head())
print("")
print("y_test head:\n", y_test.head())
print("")

# Preprocessing On-the-Fly

Per ottimizzare l'uso delle risorse, durante l'allenamento verrà eseguito un Preprocessing On-the-Fly. Questo approccio prevede il caricamento delle immagini del batch corrente, l'applicazione delle trasformazioni stabilite durante la fase di Data Augmentation e, infine, l'esecuzione del crop sull'immagine prima di passarla al modello. Questo processo permette di evitare un sovraccarico di memoria e consente una maggiore flessibilità nel trattamento dei dati.

In questa fase, il database viene ulteriormente filtrato. Vengono forniti al modello come input l'immagine trasformata e le coordinate normalizzate, mentre come output viene passato il vettore booleano delle diagnosi.

In [ ]:
scan_size = (128, 128)
batch_size = 5
steps_per_epoch = int(x_train.shape[0] / batch_size)



def preprocess_dataset(x, y):
    
    def preprocess_scan(file_path, transform):
        file_path = file_path.numpy().decode("utf-8")
        transform = transform.numpy().decode("utf-8")

        scan = get_scan(file_path)
        scan = apply_transform_to_scan(scan, transform)
        crop_scan(scan)
        
        scan = tf.convert_to_tensor(scan, dtype=tf.float32)

        if len(scan.shape) == 2:
            scan = tf.expand_dims(scan, axis=-1)
        scan = tf.image.resize(scan, [scan_size[0], scan_size[1]])
        
        max_val = tf.reduce_max(scan)
        min_val = tf.reduce_min(scan)
        if max_val == min_val:
            print("ERROR:", file_path)
            raise KeyboardInterrupt
        
        scan = (scan - min_val) / (max_val - min_val)  # Normalizzazione [0, 1]
        
        scan.set_shape([scan_size[0], scan_size[1], 1])

        return scan
    
    def preprocess_tensor(file_path, transform, coor):
        scan = tf.py_function(func=preprocess_scan, inp=[file_path, transform], Tout=tf.float32)
        scan.set_shape([scan_size[0], scan_size[1], 1])
        return scan, coor
    
    def input_generator():
        for item in x.itertuples(index=False):
            file_path, transform, coor = item
            yield file_path, transform, coor

    def output_generator():
        for item in y.itertuples(index=False):
            diagnosis = np.array(item).squeeze()
            yield diagnosis
    
    input_dataset = tf.data.Dataset.from_generator(input_generator, output_signature=(
        tf.TensorSpec(shape=(), dtype=tf.string),
        tf.TensorSpec(shape=(), dtype=tf.string),
        tf.TensorSpec(shape=(5, 2), dtype=tf.float32)
    )).map(lambda file_path, transform, coor: preprocess_tensor(file_path, transform, coor))
    
    output_dataset = tf.data.Dataset.from_generator(output_generator, output_signature=(
        tf.TensorSpec(shape=(5,), dtype=tf.bool)
    ))
    
    dataset = tf.data.Dataset.zip((input_dataset, output_dataset))
    
    return dataset



train_dataset = preprocess_dataset(x_train, y_train)
train_dataset = train_dataset.batch(batch_size).repeat()
train_dataset = train_dataset.shuffle(buffer_size=int(x_train.shape[0] / 10))
train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

test_dataset = preprocess_dataset(x_test, y_test)
test_dataset = test_dataset.batch(batch_size)

# Model per Confronto (Convolutional Model)

La struttura del modello convoluzionale è composta da due branch distinti:
* Il branch per il processamento dell'immagine, che è composto da 3 layer convoluzionali, ognuno seguito da un layer di max pooling. Termina con un layer di flatten, connesso a due layer dense.
* Il branch per il processamento delle coordinate, composto da un layer di input, seguito da un flatten e un layer dense.

I due branch convergono in un layer di concatenazione, che successivamente è seguito da un layer dense per produrre l'output finale.

In [ ]:
scan_input = Input(shape=(scan_size[0], scan_size[1], 1), name="scan_input")

scan_branch = layers.Conv2D(filters=8, kernel_size=(4, 4), strides=(1, 1), activation='relu', name="scan_0")(scan_input)
scan_branch = layers.MaxPooling2D((2, 2), name="scan_1")(scan_branch)

scan_branch = layers.Conv2D(filters=16, kernel_size=(4, 4), strides=(1, 1), activation='relu', name="scan_2")(scan_branch)
scan_branch = layers.MaxPooling2D((2, 2), name="scan_3")(scan_branch)

scan_branch = layers.Conv2D(filters=32, kernel_size=(4, 4), strides=(1, 1), activation='relu', name="scan_4")(scan_branch)
scan_branch = layers.MaxPooling2D((2, 2), name="scan_5")(scan_branch)

scan_branch = layers.Flatten(name="scan_6")(scan_branch)

scan_branch = layers.Dense(512, activation='relu', name="scan_7")(scan_branch)
scan_branch = layers.Dense(64, activation='relu', name="scan_8")(scan_branch)

# ----------------------------------------------------------------

coor_input = Input(shape=(5, 2), name="coor_input")
coor_branch = layers.Flatten(name="coor_0")(coor_input)
coor_branch = layers.Dense(10, activation='relu', name="coor_1")(coor_branch)

# ----------------------------------------------------------------

concat = layers.concatenate([scan_branch, coor_branch], name="concat")
concat_branch = layers.Dense((64 + 10), activation='relu', name="concat_0")(concat)
output = layers.Dense(5, activation='sigmoid', name="output")(concat_branch)

# ----------------------------------------------------------------

model = Model(inputs=[scan_input, coor_input], outputs=output)

In [ ]:
model.summary()
#visualkeras.layered_view(model, legend=True)

# Modello creato da Me ()

In [ ]:
# --- Scan branch (Potenziata) ---
scan_input = layers.Input(shape=(scan_size[0], scan_size[1], 1), name="scan_input")

# Blocco 1: Estrazione iniziale
x = layers.Conv2D(32, (3, 3), padding='same')(scan_input)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Blocco 2: Aumento profondità
x = layers.Conv2D(64, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Blocco 3: Cattura feature complesse (patologie)
x = layers.Conv2D(128, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Blocco 4: Astrazione finale
x = layers.Conv2D(256, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

# Pooling combinato: fondamentale per localizzare i dettagli (Max) 
# e capire il contesto (Average)
gap = layers.GlobalAveragePooling2D()(x)
gmp = layers.GlobalMaxPooling2D()(x)
scan_features = layers.concatenate([gap, gmp])
scan_features = layers.Dense(512, activation='relu')(scan_features)
scan_features = layers.Dropout(0.4)(scan_features)

# --- Coordinate branch (Ottimizzata) ---
coor_input = layers.Input(shape=(5, 2), name="coor_input")
c = layers.Flatten()(coor_input)
c = layers.Dense(128, activation='relu')(c)
c = layers.BatchNormalization()(c)
c = layers.Dense(64, activation='relu')(c)
c = layers.Dropout(0.2)(c)

# --- Fusione e Output ---
concat = layers.concatenate([scan_features, c])

# Testa decisionale profonda
f = layers.Dense(256, activation='relu')(concat)
f = layers.BatchNormalization()(f)
f = layers.Dropout(0.3)(f)
f = layers.Dense(128, activation='relu')(f)

# Output multi-label (5 patologie)
output = layers.Dense(5, activation='sigmoid', name="output")(f)

model_M = Model(inputs=[scan_input, coor_input], outputs=output)

In [ ]:
model_M.summary()

# Model Training

Per il training è stato previsto un sistema di checkpoints, che consente di salvare i progressi dell'allenamento ad ogni epoch. Questo sistema offre la possibilità di interrompere il training e riprendere successivamente dall'ultimo checkpoint salvato.

In [ ]:
checkpoint_filepath = "checkpoints/checkpoint_epoch_{epoch:02d}.keras"

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,
    save_freq="epoch",
    verbose=1
)

Per determinare l'accuratezza tra due vettori è stata sviluppata una funzione personalizzata. La funzione `vector_accuracy` binarizza le predizioni applicando una soglia e le confronta con le etichette reali. Le predizioni sono considerate corrette quando corrispondono ai valori delle etichette, una volta convertite in booleani. L'accuratezza del modello viene quindi calcolata come la media delle predizioni corrette rispetto al totale.

In [ ]:
def vector_accuracy(y_true, y_pred, threshold=0.5):
    y_pred = tf.cast(y_pred > threshold, tf.bool)
    correct_predictions = tf.equal(y_true, y_pred)
    accuracy = tf.reduce_mean(tf.cast(correct_predictions, tf.float32))
    return accuracy

Sono state selezionate solo 5 epoche di allenamento perché, dopo aver esaminato l'andamento con 10 epoche, è stato possibile osservare che le performance si stabilizzano già alla terza epoca.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=[vector_accuracy, 'precision', 'recall'])

# Imposta su "True" se desideri caricare nel modello il checkpoint "checkpoint_epoch_10.h5"
if False:
    model.load_model("checkpoints/checkpoint_epoch_10.h5")

history = model.fit(
    train_dataset,
    epochs=5,
    steps_per_epoch=steps_per_epoch,
    callbacks=[checkpoint_callback],
    validation_data=test_dataset)

In [ ]:
model_M.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=[vector_accuracy, 'precision', 'recall'])

# Imposta su "True" se desideri caricare nel modello il checkpoint "checkpoint_epoch_10.h5"
if False:
    model.load_model("/kaggle/working/checkpoints/checkpoint_epoch_10.h5")

history_M = model.fit(
    train_dataset,
    epochs=5,
    steps_per_epoch=steps_per_epoch,
    callbacks=[checkpoint_callback],
    validation_data=test_dataset)

In [ ]:
plt.figure(figsize=(12, 6))


plt.subplot(1, 2, 1)

plt.plot(history.history['vector_accuracy'], label='vector_accuracy')
plt.plot(history.history['val_vector_accuracy'], label='val_vector_accuracy')
plt.plot(history.history['precision'], label='precision')
plt.plot(history.history['val_precision'], label='val_precision')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Accuracy / Precision', fontsize=13)
plt.ylim([0.0, 1])
plt.legend(loc='upper right')

plt.subplot(1, 2, 2)

plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.ylim([0.0, 2])
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
results = model.evaluate(test_dataset)

print(f"Loss: {results[0]}")
print(f"Accuracy: {results[1]}")
print(f"Precision: {results[2]}")
print(f"Recall: {results[3]}")

In [ ]:
results = model_M.evaluate(test_dataset)

print(f"Loss: {results[0]}")
print(f"Accuracy: {results[1]}")
print(f"Precision: {results[2]}")
print(f"Recall: {results[3]}")

In [ ]:
y_true = []
y_pred = []

for x_batch, y_batch in test_dataset:
    pred = tf.cast(model_M.predict(x_batch, verbose=0) > 0.5, tf.bool)
    y_true.append(y_batch.numpy())
    y_pred.append(pred.numpy())

y_true = np.concatenate(y_true, axis=0)
y_pred = np.concatenate(y_pred, axis=0)

In [ ]:
labels = list(df_diagnosis.columns[1:6])

for i in range(5):
    cm = confusion_matrix(y_true[:, i], y_pred[:, i])
    plt.figure(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix for {labels[i]}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()